# Notebook 10 - Page-as-Image Retrieval with ColPali

This notebook introduces late interaction retrieval for document pages and shows why ColPali changes what is retrievable from visually rich documents.


## What you will learn

- how late interaction differs from single-vector retrieval
- how to compute a MaxSim-style score from token or patch embeddings
- why page-as-image retrieval helps when layout matters


In [ ]:
!pip install -q numpy pandas matplotlib
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "10_page_as_image_retrieval_with_colpali"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Implement MaxSim

Late interaction keeps multiple vectors per document instead of collapsing a full page into one average embedding.


In [ ]:
def maxsim_score(query_matrix, doc_matrix):
    similarities = query_matrix @ doc_matrix.T
    return similarities.max(axis=1).sum()

query = np.array([[0.8, 0.1, 0.1], [0.2, 0.7, 0.1]])
doc_a = np.array([[0.7, 0.2, 0.1], [0.1, 0.8, 0.1], [0.2, 0.2, 0.6]])
doc_b = np.array([[0.3, 0.4, 0.3], [0.2, 0.2, 0.6], [0.4, 0.2, 0.4]])
print("Doc A MaxSim:", round(maxsim_score(query, doc_a), 3))
print("Doc B MaxSim:", round(maxsim_score(query, doc_b), 3))


## Step 2: Compare against single-vector retrieval

Single-vector retrieval is cheaper, but it hides where the actual match happened inside the page.


In [ ]:
def dense_score(query_matrix, doc_matrix):
    query_vec = query_matrix.mean(axis=0)
    doc_vec = doc_matrix.mean(axis=0)
    return float(query_vec @ doc_vec)

comparison = pd.DataFrame([
    {"page": "A", "dense_score": dense_score(query, doc_a), "maxsim_score": maxsim_score(query, doc_a)},
    {"page": "B", "dense_score": dense_score(query, doc_b), "maxsim_score": maxsim_score(query, doc_b)},
]).round(3)
comparison


## Step 3: Rank the candidate pages

A retrieval notebook should end with a ranking, not just a theory paragraph.


In [ ]:
ranked_pages = comparison.sort_values("maxsim_score", ascending=False).reset_index(drop=True)
ranked_pages


## Exercises and extensions

1. Swap the toy matrices for real ColPali embeddings and compare the ranking.
1. Add a reranking stage that considers OCR snippets after the initial page retrieval.
